# 04 — Feature selection: from 255 inputs to a shortlist

**Purpose.** Answer one question: *which columns can we delete and still keep
the macro F1 of the full model?*

Everything else here is evidence and bookkeeping. The notebook produces two
different things, and they must not be confused:

| artifact | question it answers | routing columns? |
|---|---|---|
| `outputs/selected_features.json` | the model: the smallest set that matches the baseline | **allowed** |
| the ranking table and the themes you read from it | the report: which factors characterise age | labelled as routing |

The JSON is a *predictive* artifact. Routing columns (`*_FLAG`) are allowed in
it: they exist in the test file too, so a model may use them. The explanatory
story is a separate, later artifact.

**Read `03_recoding.ipynb` first.** This notebook starts from the recoded
frame, not from the raw text.

## How to run it

1. Run the whole notebook once with `SMOKE = True`. It takes about ten minutes
   and checks the plumbing, not the numbers. Fix every error before going on.
2. Set `SMOKE = False`, restart the kernel, run again. That is the real run:
   about two hours, or up to three if the Stage 3 check fails and the deployed
   ranking has to be rebuilt.
3. Upload the two submissions it writes and record both leaderboard scores.

**Nothing in a smoke run may be quoted.** Two folds and ten trees are not
evidence.

## 0. Setup

The imports are the whole vocabulary of the notebook. If a name is not here,
it is not used.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedShuffleSplit

from daa.config import OUTPUTS_DIR, SEED, TRAIN_PATH
from daa.data import TARGET, explicit_spec, infer_column_spec, load_test, load_train
from daa.evaluation.metrics import macro_f1
from daa.evaluation.submit import write_submission_from_result
from daa.experiment import build_pipeline, run_experiment
from daa.features.preprocess import build_preprocessor
from daa.features.recode import PARSED_COLUMNS, flag_column
from daa.features.selection import (
    load_selected_features,
    mutual_info_screen,
    ordinal_codes,
    rank_by_permutation,
    save_selection,
    subset_spec,
)
from daa.models.registry import make_model
from daa.splits import Fold, get_folds

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

if not TRAIN_PATH.exists():
    raise SystemExit("data/ is missing; this notebook needs the real files")

## 1. The smoke switch

One switch controls every stage. While `SMOKE` is true the notebook uses two
folds, ten trees, one permutation repeat, two MI shuffles, a smaller MI cut and
two K values. **Only the parameters change; the code does not.** If a cell had
to be edited to smoke-test it, that cell would be wrong.

While `SMOKE` is true, no run is saved and no estimator is fitted on all rows,
so smoke rows never reach the leaderboard. Artifacts go to `outputs/smoke/` so
the save/load path is still exercised without overwriting the real files.

The pass rule in Stage 4 is also fixed here, **before** any result is seen.

In [ ]:
SMOKE = True

X, y = load_train()
X_test, row_ids = load_test()
spec = explicit_spec(infer_column_spec(X))
INPUT_COLUMNS = list(spec.input_columns)
all_folds = get_folds(y)

# Pre-registered decision rule for Stage 4. Do not tune these after seeing the
# results; that would turn a test into a choice.
MAX_MEAN_LOSS = 0.005
MAX_REPEAT_LOSS = 0.010

if SMOKE:
    FOLDS = all_folds[:2]
    RF_TREES = 10
    PERMUTATION_REPEATS = 1
    NESTED_PERMUTATION_REPEATS = 1
    MI_SHUFFLES = 2
    MI_TOP_N = 30
    K_VALUES = (10, 30)
    NESTED_OUTER_FOLDS = 1
    SAVE = False
    FIT_FULL = False
    ARTIFACT_DIR = OUTPUTS_DIR / "smoke"
else:
    FOLDS = all_folds
    RF_TREES = 400
    PERMUTATION_REPEATS = 5
    NESTED_PERMUTATION_REPEATS = 3
    MI_SHUFFLES = 10
    MI_TOP_N = 120
    K_VALUES = (10, 15, 20, 30, 40, 60, 80)
    NESTED_OUTER_FOLDS = 5
    SAVE = True
    FIT_FULL = True
    ARTIFACT_DIR = OUTPUTS_DIR

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPEAT0_FOLDS = [fold for fold in FOLDS if fold.repeat == 0]
# The recoded columns that actually reach a model: two flags are constant and
# are dropped by the spec, so 58 candidates become 56 inputs.
RECODED_COLUMNS = [
    column
    for column in [*PARSED_COLUMNS, *(flag_column(name) for name in PARSED_COLUMNS)]
    if column in INPUT_COLUMNS
]

shares = y.value_counts(normalize=True).to_numpy()
target_entropy = float(-(shares * np.log(shares)).sum())

print(f"SMOKE = {SMOKE}")
print(f"train: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"model inputs: {len(INPUT_COLUMNS)} ({len(spec.numeric)} numeric, {len(spec.nominal)} nominal)")
print(f"recoded columns reaching the model: {len(RECODED_COLUMNS)}")
print(f"folds used: {len(FOLDS)} ({len(REPEAT0_FOLDS)} in repeat 0)")
print(f"target: {TARGET}, {y.nunique()} bands, entropy {target_entropy:.2f} nats")
print(f"artifacts: {ARTIFACT_DIR}")

## 2. Stage 0 — baselines on the recoded data

The shortlist must match a reference. This stage builds it.

- `hist_gb` with `passthrough`: `HistGradientBoostingClassifier` handles NaN
  itself, so the numeric columns need no imputation. Imputing would replace a
  real "no answer" with a median, which is a guess.
- `hist_gb` with `fit_full=True`: Stage 7 needs a fitted full-width estimator
  for the comparison submission. One extra fit now is cheaper than re-running
  this stage later.
- `rf` with `impute`: a random forest cannot handle NaN, so this variant
  imputes. This run is **not** used in any decision. It is a cheap reference
  point for the gap between model families.
- `logreg` is excluded from the whole plan. Measured: 213 s per fold at full
  width, and a linear sanity check would not change any decision here.

Both runs use all 15 frozen folds, so the numbers are comparable with every
other row on the leaderboard.

In [ ]:
baseline = run_experiment(
    "hist_gb",
    preprocessor="passthrough",
    name="hist_gb_all_recoded",
    X=X,
    y=y,
    folds=FOLDS,
    fit_full=FIT_FULL,
    save=SAVE,
)
rf_reference = run_experiment(
    "rf",
    preprocessor="impute",
    name="rf_all_recoded",
    X=X,
    y=y,
    folds=FOLDS,
    model_params={"n_estimators": RF_TREES},
    save=SAVE,
)

display(
    pd.DataFrame(
        {
            "run": [baseline.name, rf_reference.name],
            "model": ["hist_gb", "rf"],
            "macro_f1_mean": [
                baseline.metrics["macro_f1_mean"],
                rf_reference.metrics["macro_f1_mean"],
            ],
            "macro_f1_std": [
                baseline.metrics["macro_f1_std"],
                rf_reference.metrics["macro_f1_std"],
            ],
            "n_features": [baseline.n_features, rf_reference.n_features],
        }
    )
)
print(
    "reference to match: "
    f"macro F1 {baseline.metrics['macro_f1_mean']:.4f} "
    f"± {baseline.metrics['macro_f1_std']:.4f}"
)

## 3. Stage 1 — univariate screen: 255 → about 136

Mutual information (MI) measures how much one column tells us about the
target. It is cheap and it is a **filter, not a verdict**: it scores one column
at a time, so it cannot see that `HA1C` and `HA1F` are the same question, and
it cannot see two weak columns that only work together. Stage 2 exists for
that.

Every column is factorized first, so NaN becomes its own level (`-1`). MI does
not care about the order of the codes, so this is safe.

**Why the noise correction.** A shuffled target still produces a small positive
MI. On this data the noise floor reached 0.076 while the 60th real score was
0.088, so the correction is not cosmetic. One shuffle is not enough to estimate
the floor; ten are averaged and subtracted.

**Why cut at 120 and not 60.** Measured: the top-60 cut throws away 52 of the
56 recoded columns, including `BM1` (rank 77), `PE7` (97), `PE9` (101) and
`BM2` (113). Those are the columns the recoding work exists for. The top-120
cut keeps about a third of them.

**Why add every numeric column back.** Even at 120, some parsed quantities and
durations fall outside the cut. The union guarantees that all of them are
ranked in Stage 2.

**How to read an MI value.** The entropy of the target is about 2.67 nats, so
"MI = 0.74" means "28% of the uncertainty removed", not "0.74 out of 1".

In [ ]:
mi_table = mutual_info_screen(
    X[INPUT_COLUMNS],
    y,
    top_n=MI_TOP_N,
    n_shuffles=MI_SHUFFLES,
    seed=SEED,
)
screened_columns = list(dict.fromkeys([*mi_table["column"], *spec.numeric]))

print(f"MI kept {len(mi_table)} of {len(INPUT_COLUMNS)} columns (top {MI_TOP_N})")
print(f"after adding all {len(spec.numeric)} numeric columns: {len(screened_columns)}")
print(
    "recoded columns in the MI cut: "
    f"{sum(column in RECODED_COLUMNS for column in mi_table['column'])} of {len(RECODED_COLUMNS)}"
)
print(
    "recoded columns in the screened set: "
    f"{sum(column in RECODED_COLUMNS for column in screened_columns)} of {len(RECODED_COLUMNS)}"
)
best = float(mi_table["mi_adjusted"].iloc[0])
print(f"highest adjusted MI: {best:.3f} nats ({best / target_entropy:.0%} of the target entropy)")

display(mi_table.head(20))

## 4. Stage 2 — ordinal RF ranking: the full ranked list

**Why ordinal codes.** One column per raw feature, so the ranking table has one
row per named column and no dummy aggregation. It is also cheap: the forest
fits in a few seconds on about 140 columns, and the permutation step stays
narrow. The frame is already numeric, so no `Pipeline` is needed here.

**Not** the reason: permutation importance does *not* need ordinal codes. On a
`Pipeline` it shuffles the raw DataFrame column **before** the preprocessor, so
it returns one importance per raw column even when the preprocessor one-hot
encodes. Stage 3 checks whether the ordinal approximation actually matters.

**Why `n_jobs=1`.** The forest already uses every core. Measured on this
laptop, `n_jobs=-1` was slower.

**Keep the full list.** Do not truncate to 30. Stage 4 rebuilds K = 40, 60 and
80 from this ranking.

**Known bias.** Ordinal codes impose a false order on nominal columns. A tree
can still isolate one level, but it needs several splits instead of one, so a
multi-level nominal column can look weaker here than it is under one-hot.
Stage 3 measures that. Macro F1 on a single validation fold is also a noisy
scorer, which is why the standard deviation across folds is reported next to
every mean.

In [ ]:
screened_ordinal = ordinal_codes(X[screened_columns])
rf_ranker = make_model("rf", n_estimators=RF_TREES)

ranking = rank_by_permutation(
    rf_ranker,
    screened_ordinal,
    y,
    REPEAT0_FOLDS,
    n_repeats=PERMUTATION_REPEATS,
    scoring="macro_f1",
    seed=SEED,
    n_jobs=1,
)

print(f"ranked {len(ranking)} columns on {len(REPEAT0_FOLDS)} folds")
display(ranking.head(20))

top_for_plot = ranking.head(20).iloc[::-1]
fig, axis = plt.subplots(figsize=(8, 6))
axis.barh(
    top_for_plot["column"],
    top_for_plot["importance_mean"],
    xerr=top_for_plot["importance_std"],
    color="#4477aa",
)
axis.set_xlabel("mean drop in macro F1 when the column is shuffled")
axis.set_title("Top 20 columns by permutation importance (ordinal RF)")
plt.tight_layout()
plt.show()

## 5. Stage 3 — representation consistency check

Cheap insurance against the bias above. Take the top 20 columns from Stage 2
and re-rank **only those 20** with the preprocessor the model will actually
use: `hist_gb` with `passthrough`, raw columns, same folds.

**Read the number with care.** The check changes two things at once — the model
(RF → `hist_gb`) and the representation (ordinal → one-hot) — so a low
correlation cannot be blamed on the encoding alone. With n = 20 the error bars
are wide. Treat this as qualitative, not as a verdict.

**The rule is fixed in advance too.** If the correlation is high (> 0.7), the
ordinal lens is fine and the shortlist is built from the Stage 2 ranking. If it
is low, the shortlist is built from the representation the model actually uses:
the whole screened set is re-ranked with `hist_gb` + `passthrough`. That second
ranking costs about half an hour and only runs when the check fails.

This is a pre-registered branch, not a choice made after seeing the score. The
only thing the number decides is *which ranking was already declared to win*.

In [ ]:
TOP_FOR_CHECK = min(20, len(ranking))
top_columns = list(ranking["column"].head(TOP_FOR_CHECK))

check_pipeline = build_pipeline(
    build_preprocessor(subset_spec(spec, top_columns), variant="passthrough"),
    "hist_gb",
)
deployed_ranking = rank_by_permutation(
    check_pipeline,
    X[top_columns],
    y,
    REPEAT0_FOLDS,
    n_repeats=PERMUTATION_REPEATS,
    scoring="macro_f1",
    seed=SEED,
    n_jobs=1,
)

comparison = (
    ranking.set_index("column")[["importance_mean"]]
    .rename(columns={"importance_mean": "ordinal_rf"})
    .join(
        deployed_ranking.set_index("column")[["importance_mean"]]
        .rename(columns={"importance_mean": "deployed_hist_gb"}),
        how="inner",
    )
    .loc[top_columns]
)
comparison["rank_ordinal_rf"] = (
    comparison["ordinal_rf"].rank(ascending=False, method="min").astype(int)
)
comparison["rank_deployed"] = (
    comparison["deployed_hist_gb"].rank(ascending=False, method="min").astype(int)
)
spearman = float(spearmanr(comparison["ordinal_rf"], comparison["deployed_hist_gb"]).statistic)

display(comparison)
print(f"Spearman correlation between the two rankings: {spearman:.3f}")

if spearman > 0.7:
    ranking_for_selection = ranking
    ranking_source = "stage2_ordinal_rf"
    ranker_uses_ordinal = True
    print("High: the ordinal lens is fine. The shortlist uses the Stage 2 ranking.")
else:
    print("Low: re-ranking the whole screened set with the deployed pipeline.")
    print("This takes about half an hour; it is the pre-registered fallback.")
    deployed_pipeline = build_pipeline(
        build_preprocessor(subset_spec(spec, screened_columns), variant="passthrough"),
        "hist_gb",
    )
    ranking_for_selection = rank_by_permutation(
        deployed_pipeline,
        X[screened_columns],
        y,
        REPEAT0_FOLDS,
        n_repeats=PERMUTATION_REPEATS,
        scoring="macro_f1",
        seed=SEED,
        n_jobs=1,
    )
    ranking_source = "stage3_deployed_hist_gb"
    ranker_uses_ordinal = False
    display(ranking_for_selection.head(20))

print(f"ranking used for the shortlist: {ranking_source}")

## 6. Stage 4 — K sweep with a rule fixed in advance

Do **not** look at the results first and then decide what "within noise" means.
The rule is fixed in the parameter cell:

> K passes if the mean paired difference over the folds is **at most 0.005**
> and no single repeat shows a loss worse than 0.010.

The comparison is **paired**: for each fold, baseline minus K, on the same
rows. The 15 folds are not independent — three repeats score the same rows with
different splits — so the standard deviation across folds is descriptive, not
a confidence interval. Do not compute "mean ± 2 SE" from it. The paired
differences are the honest comparison, and the per-repeat check is the guard.

**Pre-registered fallback.** If no K passes, not even K = 80, the shortlist is
the full input set: `selected_features.json` records the full list and the
report says that feature selection did not survive validation. That is decided
now, with the rule, not after seeing the results.

**Expected shape.** The archive measured, on `data_v1` and with *accuracy*,
that top-20 lost 4.3 percentage points. Macro F1 weights the rare bands more,
so the elbow may sit at 30–40. If K = 80 passes and K = 60 does not, that is a
real result: report it and accept the larger set. A larger K is not a failure.

In [ ]:
sweep_rows = []
for k in K_VALUES:
    k_columns = list(ranking_for_selection["column"].head(k))
    result = run_experiment(
        "hist_gb",
        preprocessor="passthrough",
        name=f"hist_gb_k{k}",
        X=X,
        y=y,
        folds=FOLDS,
        columns=k_columns,
        feature_set=f"top{k}",
        save=SAVE,
    )
    differences = baseline.fold_scores["macro_f1"] - result.fold_scores["macro_f1"]
    per_repeat = differences.groupby(result.fold_scores["repeat"]).mean()
    sweep_rows.append(
        {
            "k": k,
            "macro_f1_mean": result.metrics["macro_f1_mean"],
            "macro_f1_std": result.metrics["macro_f1_std"],
            "paired_difference_mean": float(differences.mean()),
            "worst_repeat_difference": float(per_repeat.min()),
            "passes": bool(
                differences.mean() <= MAX_MEAN_LOSS and per_repeat.min() >= -MAX_REPEAT_LOSS
            ),
            "run_id": result.run_id,
        }
    )

sweep_table = pd.DataFrame(sweep_rows)
display(sweep_table)

passing = sweep_table[sweep_table["passes"]]
if len(passing):
    chosen_k = int(passing["k"].min())
    selected_columns = list(ranking_for_selection["column"].head(chosen_k))
    decision = f"K = {chosen_k} is the smallest set that passes the rule."
else:
    chosen_k = len(INPUT_COLUMNS)
    selected_columns = list(INPUT_COLUMNS)
    decision = "No K passes. Fallback: the shortlist is the full input set."
print(decision)

fig, axis = plt.subplots(figsize=(7, 4))
axis.errorbar(
    sweep_table["k"],
    sweep_table["macro_f1_mean"],
    yerr=sweep_table["macro_f1_std"],
    marker="o",
    capsize=3,
    label="shortlist",
)
axis.axhline(
    baseline.metrics["macro_f1_mean"],
    color="#cc3311",
    linestyle="--",
    label="baseline (all inputs)",
)
axis.set_xlabel("K (number of columns kept)")
axis.set_ylabel("macro F1")
axis.set_title("K sweep: does a smaller set keep the baseline?")
axis.legend()
plt.tight_layout()
plt.show()

## 7. Stage 5 — nested-selection honesty check

Stages 1 and 2 look at the labels of every row. With 5-fold CV, every row is in
some validation fold, so *every* row's label influenced the choice of columns.
Reporting the reduced model's CV score as if selection were free overstates it.
The frozen folds protect the *preprocessor* from leaking; they do not protect
you from looking at validation scores while choosing.

So run the whole selection procedure inside each outer fold:

1. Take the outer fold's **training rows only**.
2. Carve an inner 20% holdout out of them. The ranker never sees the holdout
   rows while fitting.
3. Run Stage 1 (MI) on the remaining 80%, fit the ranker there, and rank by
   permutation **on the holdout**. The permutation is then out-of-sample,
   exactly like Stage 2. Fitting on all outer training rows and permuting on a
   subset of them would be in-sample; that inflates high-cardinality columns
   and contaminates the gap.
4. Take the top K, fit `hist_gb` on the outer training rows, score on the outer
   validation fold.

The ranker is the one the shortlist actually came from: the ordinal RF when
Stage 3 passed, the deployed `hist_gb` pipeline when it did not. Mirroring the
ranker keeps the gap a measure of **selection optimism**, not of disagreement
between two different rankers.

That number estimates **the procedure**, not just the model. The gap to Stage
4's number for the same K is the selection optimism. It goes in the report as a
measured quantity, not as a caveat.

In [ ]:
nested_rows = []
for outer in REPEAT0_FOLDS[:NESTED_OUTER_FOLDS]:
    outer_train = X.iloc[outer.train_idx]
    outer_train_target = y.iloc[outer.train_idx]

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    fit_positions, holdout_positions = next(
        splitter.split(np.zeros(len(outer_train)), outer_train_target)
    )

    fit_frame = outer_train.iloc[fit_positions]
    fit_target = outer_train_target.iloc[fit_positions]
    holdout_frame = outer_train.iloc[holdout_positions]
    holdout_target = outer_train_target.iloc[holdout_positions]
    combined_frame = pd.concat([fit_frame, holdout_frame])
    combined_target = pd.concat([fit_target, holdout_target])

    inner_mi = mutual_info_screen(
        fit_frame[INPUT_COLUMNS],
        fit_target,
        top_n=MI_TOP_N,
        n_shuffles=MI_SHUFFLES,
        seed=SEED,
    )
    inner_screened = list(dict.fromkeys([*inner_mi["column"], *spec.numeric]))

    inner_fold = Fold(
        repeat=outer.repeat,
        fold=outer.fold,
        train_idx=np.arange(len(fit_frame)),
        val_idx=np.arange(len(fit_frame), len(combined_frame)),
    )
    if ranker_uses_ordinal:
        inner_ranker = make_model("rf", n_estimators=RF_TREES)
        inner_frame = ordinal_codes(combined_frame[inner_screened])
    else:
        inner_ranker = build_pipeline(
            build_preprocessor(subset_spec(spec, inner_screened), variant="passthrough"),
            "hist_gb",
        )
        inner_frame = combined_frame[inner_screened]

    inner_ranking = rank_by_permutation(
        inner_ranker,
        inner_frame,
        combined_target,
        [inner_fold],
        n_repeats=NESTED_PERMUTATION_REPEATS,
        scoring="macro_f1",
        seed=SEED,
        n_jobs=1,
    )
    inner_top = list(inner_ranking["column"].head(chosen_k))

    nested_pipeline = build_pipeline(
        build_preprocessor(subset_spec(spec, inner_top), variant="passthrough"),
        "hist_gb",
    )
    nested_pipeline.fit(outer_train, outer_train_target)
    predictions = nested_pipeline.predict(X.iloc[outer.val_idx])
    nested_rows.append(
        {
            "fold": outer.fold,
            "k": chosen_k,
            "macro_f1": macro_f1(y.iloc[outer.val_idx], predictions),
            "columns_differing_from_stage4": len(set(inner_top) ^ set(selected_columns)),
        }
    )

nested_table = pd.DataFrame(nested_rows)
display(nested_table)

stage4_score = sweep_table.loc[sweep_table["k"] == chosen_k, "macro_f1_mean"]
if len(stage4_score):
    optimism = float(stage4_score.iloc[0] - nested_table["macro_f1"].mean())
    print(f"selection optimism (Stage 4 minus Stage 5): {optimism:+.4f}")
else:
    print("Stage 4 did not score this K (fallback path); no gap to report.")

## 8. Stage 6 — persist the shortlist

Notebook outputs are stripped by the pre-commit hook and `outputs/*.png` is
gitignored, so a table kept in the notebook is not a durable artifact. These
files are:

| file | contents |
|---|---|
| `feature_selection_ranking.csv` | full Stage 2 ranked list: column, mean, std, rank |
| `feature_selection_ranking_deployed.csv` | the deployed ranking, only when Stage 3 failed and rebuilt it |
| `feature_selection_k_sweep.csv` | K, macro F1, paired difference, pass flag |
| `selected_features.json` | ordered list, K, method, ranking source, baseline comparison, run ids, git state, seed, fold spec, fingerprints |

The loader checks that every name still exists, that the list is unique, and
that the fingerprint matches. A shortlist that no longer fits the data must
fail loudly, not silently drop a column.

In [ ]:
RANKING_PATH = ARTIFACT_DIR / "feature_selection_ranking.csv"
DEPLOYED_RANKING_PATH = ARTIFACT_DIR / "feature_selection_ranking_deployed.csv"
SWEEP_PATH = ARTIFACT_DIR / "feature_selection_k_sweep.csv"
SELECTION_PATH = ARTIFACT_DIR / "selected_features.json"

ranking.to_csv(RANKING_PATH, index=False)
sweep_table.to_csv(SWEEP_PATH, index=False)
save_selection(
    SELECTION_PATH,
    selected_columns,
    spec=spec,
    target=y,
    metadata={
        "method": "mutual-info screen, then permutation importance",
        "ranking_source": ranking_source,
        "stage3_spearman": spearman,
        "decision": decision,
        "smoke": SMOKE,
        "baseline_run_id": baseline.run_id,
        "baseline_macro_f1": baseline.metrics["macro_f1_mean"],
        "sweep_run_ids": list(sweep_table["run_id"]),
        "folds": {"n_splits": 5, "n_repeats": 3, "seed": SEED, "source": "splits/folds.npz"},
    },
)
print(f"wrote {RANKING_PATH}")
if ranking_source != "stage2_ordinal_rf":
    ranking_for_selection.to_csv(DEPLOYED_RANKING_PATH, index=False)
    print(f"wrote {DEPLOYED_RANKING_PATH}")
print(f"wrote {SWEEP_PATH}")
print(f"wrote {SELECTION_PATH}")

reloaded = load_selected_features(SELECTION_PATH, spec, target=y)
assert reloaded.columns == tuple(selected_columns)
print(f"reloaded {reloaded.k} columns; fingerprint {reloaded.fingerprint[:12]}")

## 9. Stage 7 — one submission comparison

Stages 4 and 5 cannot tell you whether the shortlist generalises. Only the
leaderboard can, and the competition allows three submissions per day.

1. Re-run the chosen K with `fit_full=True` and write a submission.
2. Write a submission from Stage 0's full-width estimator. It was fitted with
   `fit_full=True`, so no re-run is needed.
3. Upload both and record both scores. That spends two of the three daily
   submissions.

If the shortlist is clearly worse on the leaderboard but equal in CV, that is
the selection optimism showing up, and it is a finding worth reporting.

In [ ]:
selected_result = run_experiment(
    "hist_gb",
    preprocessor="passthrough",
    name=f"hist_gb_selected_k{chosen_k}",
    X=X,
    y=y,
    folds=FOLDS,
    columns=selected_columns,
    feature_set=f"selected_k{chosen_k}",
    fit_full=FIT_FULL,
    save=SAVE,
)

display(
    pd.DataFrame(
        {
            "run": [baseline.name, selected_result.name],
            "macro_f1_mean": [
                baseline.metrics["macro_f1_mean"],
                selected_result.metrics["macro_f1_mean"],
            ],
            "macro_f1_std": [
                baseline.metrics["macro_f1_std"],
                selected_result.metrics["macro_f1_std"],
            ],
            "n_features": [baseline.n_features, selected_result.n_features],
        }
    )
)

if SAVE:
    selected_path = write_submission_from_result(
        selected_result,
        X_test,
        row_ids,
        OUTPUTS_DIR / f"submission_selected_k{chosen_k}.csv",
    )
    full_path = write_submission_from_result(
        baseline,
        X_test,
        row_ids,
        OUTPUTS_DIR / "submission_all_features.csv",
    )
    print(f"wrote {selected_path}")
    print(f"wrote {full_path}")
    print("upload both and record the two leaderboard scores")
else:
    print("SMOKE: submissions are not written (fit_full is off)")

## 10. Caveats to state in the report

1. **Selection optimism is measured, not hand-waved.** Stage 5 reports the gap
   between the nested score and the Stage 4 score. Quote it.
2. **The shortlist will be redundant.** 13 of the MI top-20 are `HA1A`–`HA1G`,
   `PC1A`–`PC1E` and `IN15`. Their plain `Não aplicável` level is 100% for
   under-55 and 0% for 55+, so one split at 55 captures them. They are one
   age-eligibility signal, not thirteen health findings. Permutation importance
   also splits credit among them, so a single member can look weak while the
   group is strong. Report themes, not thirteen discoveries.
3. **27 usable flags.** `HHNBPERS_COD_FLAG` and `HHNBPERS_0_13_COD_FLAG` are
   constant and dropped. Sentinels and routing flags are legitimate signal;
   they exist in the test file too.
4. **The Cramér's V cross-check is qualitative.** It was computed on the
   pre-recode frame and excluded high-cardinality columns. Broad disagreement
   with MI is expected, not evidence of a bug.
5. **The band-0 dispute does not block selection.** The ranking treats labels
   as codes; see `TODO.md`.
6. **The archived `data_v1` numbers are accuracy.** Top-20 lost 4.3 percentage
   points there. Every comparison here is macro F1.
7. **Ordinal codes impose a false order** on nominal columns. Stage 3 measures
   whether that changes the ranking.
8. **Macro-F1 permutation is noisier than accuracy**, especially for the rare
   bands. That is why every mean is reported with a standard deviation across
   folds.

## 11. What to write down

- The baseline: macro F1 mean ± std, and the run id.
- The chosen K, the paired differences per fold, and the pass rule that was
  fixed in advance.
- The selection optimism from Stage 5.
- The Spearman correlation from Stage 3, and **which ranking the shortlist came
  from** (`stage2_ordinal_rf` or `stage3_deployed_hist_gb`). A low correlation
  is a result: it says the ordinal lens changed the answer.
- The two leaderboard scores from Stage 7.
- The themes, not the thirteen `HA1*`/`PC1*` columns as separate findings.

`outputs/selected_features.json` is the artifact the model uses. The themes
table is the artifact the report uses. They answer different questions.